##Defining kafka connector to the endpoint

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

dbutils.widgets.text(
    "namespace",
    "evhua5816bd"
)

NAMESPACE = dbutils.widgets.get("namespace")

EVENT_HUB_NAME = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-name"
)

EH_CONN_STR = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-connstr-listen"
)

kafka_options = {
    "kafka.bootstrap.servers":
        f"{NAMESPACE}.servicebus.windows.net:9093",
    "subscribe":
        EVENT_HUB_NAME,
    "kafka.security.protocol":
        "SASL_SSL",
    "kafka.sasl.mechanism":
        "PLAIN",
    "kafka.sasl.jaas.config":
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="$ConnectionString" '
        f'password="{EH_CONN_STR}";',
    "startingOffsets": "latest"
}

##Consumer creation, parsing of Json and saving to Delta

In [0]:
events = (
    spark.readStream
        .format("kafka")
        .options(**kafka_options)
        .option("startingOffsets", "latest")
        .load()
)

json_events = events.select(
    F.col("value").cast("string").alias("json")
)

schema = StructType([
    StructField("id", StringType()),
    StructField("properties", StructType([
        StructField("mag", DoubleType()),
        StructField("place", StringType()),
        StructField("time", LongType()),
        StructField("tsunami", IntegerType()),
        StructField("alert", StringType())
    ])),
    StructField("geometry", StructType([
        StructField("coordinates", ArrayType(DoubleType()))
    ]))
])

parsed = (
    json_events.select(
        F.from_json(F.col("json"), schema).alias("data")
    )
)

earthquakes = parsed.select(
    F.col("data.id").alias("earthquake_id"),
    F.col("data.properties.mag").alias("magnitude"),
    F.col("data.properties.place").alias("place"),
    F.col("data.properties.time").alias("event_time"),
    F.col("data.properties.tsunami").alias("tsunami"),
    F.col("data.properties.alert").alias("alert"),
    F.col("data.geometry.coordinates")[0].alias("longitude"),
    F.col("data.geometry.coordinates")[1].alias("latitude"),
    F.col("data.geometry.coordinates")[2].alias("depth")
)

query = (
    earthquakes.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://artgulidov@dlsua5816bd.dfs.core.windows.net/streaming/streaming_metadata/eventhub_checkpoint/"
        )
        .trigger(processingTime="5 seconds")
        .toTable(
            "dbr_dev_ua5816bd.gulidov_artem_staging.streaming_earthquakes_staging_eventhub"
        )
)

In [0]:
print(query.lastProgress)